# 084 — Serving, batching y cachés

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

- **Dos fases**: prefill (paralelo, limitado por cómputo, define TTFT) y decode
  (token a token, limitado por ancho de banda de memoria, define TPOT).
- **KV cache**: guarda K y V por capa/cabeza para no recomputar atención; memoria
  = 2·capas·kv_heads·d_head·seq·bytes → ~0,5 MB/token en un 7B FP16 (2,15 GB a
  4 096 tokens). GQA/MQA la reducen 4–8×.
- **Continuous batching**: el scheduler decide por iteración qué secuencias entran
  y salen del lote; elimina la espera del batching estático.
- **PagedAttention**: KV cache en bloques no contiguos con tabla de páginas
  (idea de memoria virtual); desperdicio <4 % y prefijos compartidos
  copy-on-write → 2–4× throughput (vLLM).

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("observability", seed=84)
show(result)


## Reflexión

1. ¿Por qué el decode está limitado por ancho de banda y no por FLOPs, y qué
   implica para elegir GPU de serving frente a GPU de entrenamiento?
2. Un prompt de sistema de 2 000 tokens compartido por todos los usuarios: ¿qué
   mecanismo de esta clase lo convierte en casi gratis y por qué?
3. Si tu p99 de TPOT empeora al subir el tamaño máximo de lote, ¿qué compromiso
   estás viendo y cómo fijarías el límite?